# 실습 ③ 다중 벡터 검색기 + 최종 체인 — 연습본


색인 카드(요약문)를 **서랍**에 꽂고, 원본을 **서가**에 보관해 연결합니다. 오늘 수업의 마지막 조각입니다.

---

## 오늘 만드는 것

```
질문 ──▶ 요약으로 검색 ──▶ 원본 꺼내기 ──▶ 멀티모달 LLM ──▶ 답변
         (색인 카드)       (진짜 책)        (눈 달린 모델)
```

## 진행 방법

1. 셀을 **위에서부터 순서대로** 실행합니다 (`Shift + Enter`)
2. 코드 셀의 **주석을 따라 코드를 직접 작성**합니다 (`# TODO` 는 핵심 포인트)
3. 각 단계의 **확인 포인트**를 통과하면 다음으로 넘어갑니다

## ⚠️ 마지막 8번 셀을 꼭 하세요

**Before/After 비교**는 과제 채점에서 7점이고, 포트폴리오에서 가장 강력한 자료입니다. 시간이 모자라면 9번(번역 체인)을 건너뛰고 8번을 하세요.

## 0. 결과 복원

지금까지 만든 모든 재료를 한 번에 불러옵니다.

> ⚠️ `FileNotFoundError`가 나면 앞 교시의 저장 셀을 실행하지 않은 것입니다.

In [1]:
# ── 0. 결과 복원 ──────────────────────────────────────────
# 1) 필요한 모듈 import: os, re, time, uuid, pickle, pathlib.Path
import os               # 환경변수 읽기: os.getenv("OPENAI_API_KEY") 로 API 키 존재 확인
import re               # 정규식: 검색 결과에서 base64 이미지 판별, 텍스트 정리 등 문자열 패턴 처리/정규표현식으로 문자열 패턴, 만든 검사, 검색, 치환 
import time             # 실행 시간 측정: time.time() 으로 임베딩·적재 소요 시간 계산/현재 시각확인,시간 측정, 대기(체인 답변 생성 시간 측정)
import uuid             # 고유 ID 생성: uuid4() 로 doc_ids 를 만들어 요약(벡터스토어) ↔ 원본(docstore) 연결/uuid.uuid4() 요약본, 원본을 doc_id로 페어링
import pickle           # 이전 노트북에서 저장한 extracted.pkl / summaries.pkl 을 파이썬 객체로 복원
from pathlib import Path  # OS 무관한 경로 처리: .env 위치 탐색, PROJECT_ROOT / "data" / "processed" 경로 조합/경로를 객체로 다룰때 사용하는 기능
# 2) python-dotenv 의 load_dotenv import

from dotenv import load_dotenv

# 3) .env 가 있는 폴더를 프로젝트 루트로 자동 탐색
#    - Path.cwd() 와 그 상위 폴더(parents)를 차례로 확인해 (p / ".env") 가 존재하는 첫 경로를 PROJECT_ROOT 로 지정
#    - 찾지 못하면 Path.cwd() 를 사용
#    - PROJECT_ROOT 출력 → c:\workspaces\multimodal-rag 이면 정상
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists()),
    Path.cwd(),
)

print("프로젝트 루트 :", PROJECT_ROOT)

# 4) PROJECT_ROOT 의 .env 파일 로드 (load_dotenv)
load_dotenv(PROJECT_ROOT / ".env")
# 5) OPENAI_API_KEY 가 없으면 ValueError 발생시키기
try: 
    openai_api_key = os.environ["OPENAI_API_KEY"]
    if not openai_api_key.strip():
        raise ValueError("OPENAI_API_KEY 값이 비어 있습니다. .env파일 확인해 주세요")
except KeyError: 
    raise ValueError("OPENAI_API_KEY 키가 없습니다. .env파일을 확인해 주새요")

#안전한 종료를 하기위해 예외처리를 한다.

# 6) processed 경로 = PROJECT_ROOT / "data" / "processed"
processed = PROJECT_ROOT / "data" / "processed"


# 7) 추출 결과 복원: processed/"extracted.pkl" 을 pickle 로 읽기 ("rb")
#    - texts, tables, text_2k_token 꺼내기
with open(processed/"extracted.pkl","rb") as f:
    doc1 = pickle.load(f)
    texts, tables, text_2k_token = doc1["texts"], doc1["tables"], doc1["texts_2k_token"]

# 8) 요약 결과 복원: processed/"summaries.pkl" 을 pickle 로 읽기
#    - text_summaries, table_summaries, image_summaries, img_base64_list 꺼내기
with open(processed/"summaries.pkl","rb") as f1:
    doc2 = pickle.load(f1)
    text_summaries = doc2["text_summaries"]
    table_summaries = doc2["table_summaries"]
    image_summaries = doc2["image_summaries"]
    img_base64_list = doc2["img_base64_list"]

# 9) "복원 완료" 와 함께 텍스트·표·이미지의 원본 수 / 요약 수 출력
print("복원 완료")
print(f" 텍스트 원본 {len(text_2k_token):3d} 건 / 요약 {len(text_summaries):3d} 건")
print(f" 테이블 원본 {len(tables):3d} 건 / 요약 {len(table_summaries):3d} 건")
print(f" 이미지 원본 {len(img_base64_list):3d} 건 / 요약 {len(image_summaries):3d} 건")

프로젝트 루트 : /Users/yejinddo/Desktop/workspaces/multimodal-rag
복원 완료
 텍스트 원본  27 건 / 요약  27 건
 테이블 원본  35 건 / 요약  35 건
 이미지 원본   6 건 / 요약   6 건


> ✅ **확인 포인트:** 텍스트·표·이미지의 원본 수와 요약 수가 각각 같아야 합니다. (표는 0건일 수 있습니다)

## 1. 다중 벡터 검색기의 구조

> **비유:** **도서관의 색인 카드**입니다. 사서는 책을 다 읽고 찾지 않습니다. 요약이 적힌 **색인 카드로 검색**하고, 찾으면 **서가에서 진짜 책을 꺼내** 건네줍니다.

| 구성 요소 | 비유 | 들어가는 것 | 역할 |
| --- | --- | --- | --- |
| `vectorstore` (Chroma) | 색인 카드 서랍 | 요약문 임베딩 | **검색** |
| `docstore` (InMemoryStore) | 서가 | 원본 텍스트·표·Base64 이미지 | **전달** |
| `id_key` | 청구기호 | `doc_id` | 둘을 **연결** |

오늘 배우는 것의 핵심은 한 줄입니다.

> **요약과 원본에 같은 번호표를 붙인다.**

## 2. 검색기 만들기

아래 셀의 주석을 따라 코드를 작성하세요.

> `InMemoryStore`는 메모리에만 저장됩니다. 커널을 재시작하면 사라지므로 실제 서비스에서는 파일이나 Redis를 씁니다.

In [2]:
# ── 2. 검색기 만들기 ──────────────────────────────────────
# langchain 1.x 에서 두 클래스의 위치가 바뀌었습니다.
# 어느 버전을 설치했든 동작하도록 두 경로를 모두 시도합니다. (try / except ImportError)

# 1) MultiVectorRetriever import
#    - langchain 1.x : langchain_classic.retrievers.multi_vector
#    - langchain 0.3 : langchain.retrievers.multi_vector
try:
    from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
except ImportError:
    print("MultiVectorRetriever 클래스가 필요합니다.")

# 2) InMemoryStore import 서점
#    - langchain 1.x : langchain_core.stores
#    - langchain 0.3 : langchain.storage
try:
    from langchain_core.stores import InMemoryStore
except:
    print("InMemoryStore 클래스가 필요합니다.")
# 3) Chroma(langchain_chroma), OpenAIEmbeddings(langchain_openai) import 임베딩 엔진

from langchain_chroma import Chroma 
from langchain_openai import OpenAIEmbeddings

# 4) 요약문이 들어갈 벡터 DB (= 색인 카드 서랍)
#    - collection_name="multimodal_rag"
#    - embedding_function = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(
    collection_name="multimodal_rag", embedding_function = OpenAIEmbeddings(model="text-embedding-3-small")

)

# 5) 원본이 들어갈 저장소 (= 서가) : InMemoryStore()
store =InMemoryStore()
# 6) 연결 키(= 청구기호) : id_key = "doc_id"
id_key = "doc_id"

# 7) MultiVectorRetriever(vectorstore=..., docstore=..., id_key=...) 생성

retriever = MultiVectorRetriever(
    vectorstore = vectorstore, docstore= store, id_key = id_key

)

# 8) "검색기 생성 완료" 출력

print("검색기 생성 완료")


검색기 생성 완료


## 3. 요약과 원본을 같은 ID로 짝짓기

**TODO 1~2.** 주석을 따라 함수를 작성하세요.

💡 **힌트:** 위 표를 보세요.
- 요약문은 **검색**에 쓰입니다 → 어디에 넣어야 할까요?
- 원본은 **전달**에 쓰입니다 → 어디에 넣어야 할까요?

> ⚠️ **이 셀을 두 번 실행하면 데이터가 중복됩니다.** 다시 돌리려면 2번 셀부터 재실행하세요.

In [3]:
# ── 3. 요약과 원본을 같은 ID로 짝짓기 ─────────────────────
# 1) langchain_core.documents 의 Document import
from langchain_core.documents import Document as _Doc

# 2) add_documents(retriever, doc_summaries, doc_contents) 함수 정의
#    """요약본은 벡터 DB에, 원본은 docstore에 '같은 id'로 저장합니다."""
#    - 원본 개수만큼 uuid4 로 doc_ids 생성 (문자열로 변환)
#    - 요약문마다 Document(page_content=요약, metadata={id_key: doc_ids[i]}) 생성

def add_documents(retriever, doc_summaries, doc_contents):
    doc_ids = []
    # for _ in doc_contents:
    #     new_id = str(uuid.uuid4())
    #     doc_ids.append(new_id)
    doc_ids = [str(uuid.uuid4()) for _ in doc_contents]

    summaries_docs = [
        _Doc(page_content= s, metadata= {id_key: doc_ids[i]})
        for i, s in enumerate(doc_summaries)


    ]

#    - TODO 1: 요약본은 어디에 넣을까요? (vectorstore / docstore)
#      → retriever.vectorstore.add_documents(...)

    retriever.vectorstore.add_documents(summaries_docs)

#    - TODO 2: 원본은 어디에 넣을까요? set 은 집합 중복을 허용하지 않음
    retriever.docstore.mset(list(zip(doc_ids, doc_contents)))


# 3) 데이터가 있을 때만 각각 저장
#    - text_summaries  ↔ text_2k_token
#    - table_summaries ↔ tables
#    - image_summaries ↔ img_base64_list

if text_summaries: 
    add_documents(retriever, text_summaries, text_2k_token)
if table_summaries:
    add_documents(retriever, table_summaries, tables)
if image_summaries:
    add_documents(retriever, image_summaries, img_base64_list)


# 4) "색인 완료" 와 벡터 DB에 저장된 요약 수 출력 (vectorstore.get()["ids"] 의 길이)
# ⚠️ 이 셀을 두 번 실행하면 데이터가 중복됩니다.

print("색인 완료")
print(" 벡터 DB에 저장된 요약 수:", len(vectorstore.get()["ids"]))

색인 완료
 벡터 DB에 저장된 요약 수: 68


## 4. 검색 테스트 — 요약으로 찾고 원본을 받는다

**TODO 3.** 본인 PDF에 맞는 질문을 적으세요.

💡 **힌트:** **그래프나 표를 봐야만** 답할 수 있는 질문이어야 합니다. 텍스트에도 똑같이 적혀 있는 내용을 물으면 나중에 Before/After 차이가 안 납니다.

> 좋은 예: "그래프에 나타난 2023년 추세를 설명해줘"
> 나쁜 예: "이 문서는 무엇에 관한 것인가요?"

In [4]:
# ── 4. 검색 테스트 ────────────────────────────────────────
# TODO 3: 그래프나 표를 봐야만 답할 수 있는 질문을 query 에 적으세요

query = "과거 데이터 모의실험에서 상환 차수별 횟수와 빈도율은 어떻게 되나요? 1차 상환 비율은 몇 %인가요?"

# 1) retriever.invoke(query) 결과를 found 에 저장
found = retriever.invoke(query)

# 1. 질문을 임베딩으로 바꾼다.
# 2. vectorstore에서 질문과 가장 비슷한 요약문을 찾는다.(Chroma)
# 3. 찾은 요약본을 metadata["doc_id"]를 읽는다.
# 4. doc_id(번호표) docstore에서 원본을 꺼내 반환(found로 넘겨준다)한다.(InMemory Store)


# 2) 검색된 원본 수 출력
print("검색된 원본 수 :", len(found))
print()

# 3) found 를 순회하며
#    - content = doc.page_content (Document 가 아니면 doc 그대로)
#    - 정규식 "^[A-Za-z0-9+/]{100,}={0,2}$" 에 맞으면 "이미지(Base64)", 아니면 "텍스트/표"
#    - 종류, 길이, 앞 120자(줄바꿈 제거) 출력

for i, doc in enumerate(found):
    content = doc.page_content if hasattr(doc, "page_content") else doc
    is_image = bool(re.match("^[A-Za-z0-9+/]{100,}={0,2}$", content))
    #base64에서 쓰는 문자만 100개 이상 이어지고 , 끝에 = 이 0~2 개 붙은 문자열은 이미지다. 
    kind = "이미지(Base64)" if is_image else "텍스트/표"
    print(f"[{i}] {kind} — 길이 {len(content):,}자")
    print("   ", content[:120].replace("\n", " "), "...")
    print()


검색된 원본 수 : 4

[0] 이미지(Base64) — 길이 142,416자
    /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAxNDQ0Hyc5PTgyPC4zNDL/ ...

[1] 텍스트/표 — 길이 971자
    주1) 위의 표와 그래프는 분석시작일인 [2003년 01월 02일]부터 [2023년 11월 02일]까지의 해당 상품의 일 별 수익률 변화를 분석한 자료입니다.  주2) 분석시작일부터 [2023년 11월 02일]까지는 ...

[2] 텍스트/표 — 길이 269자
    차수 자동조기상환평가일 조건 상환금액(USD)(세전) 1차 2027년 05월 12일 1차 자동조기상환조건 충족시 액면금액 × 103.05% 2차 2027년 11월 12일 2차 자동조기상환조건 충족시 액면금액 × 10 ...

[3] 텍스트/표 — 길이 224자
    차수 자동조기상환 발생조건 1차 기초자산의 1차 자동조기상환평가가격이 최초기준가격의 80%이상인 경우 2차 기초자산의 2차 자동조기상환평가가격이 최초기준가격의 80%이상인 경우 3차 기초자산의 3차 자동조기상환평가가 ...



> ✅ **확인 포인트:** 검색 결과에 **이미지(Base64)가 최소 1건** 들어 있어야 합니다.
>
> 하나도 없다면 이미지 요약의 품질이 낮아 검색이 안 되는 것입니다. 6교시로 돌아가 프롬프트를 보강하세요.

> 💡 **여기가 오늘의 핵심 장면입니다.** 검색어와 글자가 하나도 안 겹치는 원본이 나왔죠? 요약문을 거쳐서 찾았기 때문입니다.

## 5. 검색 결과를 텍스트와 이미지로 분리

멀티모달 LLM에 넘기려면 둘을 다른 형식으로 포장해야 합니다. Base64 문자열의 **생김새**로 구분합니다.

아래 셀의 주석을 따라 코드를 작성하세요.

In [5]:
# ── 5. 검색 결과를 텍스트와 이미지로 분리 ─────────────────
# 1) langchain_core.documents 의 Document import
from langchain_core.documents import Document

# 2) split_image_text_types(docs) 함수 정의
#    """검색 결과를 Base64 이미지와 텍스트로 나눕니다."""
#    - b64_images, texts_out 빈 리스트 준비
#    - Document 면 page_content 를 꺼내 문자열로 만들기
#    - Base64 문자열은 영문/숫자/+/= 만으로 길게 이어집니다
#      → 정규식 "^[A-Za-z0-9+/]{100,}={0,2}$" 매칭 시 이미지, 아니면 텍스트
#    - {"images": b64_images, "texts": texts_out} 반환
def split_image_text_types(docs):
    b64_images = []
    texts_out = []

    for doc in docs:
        doc.page_content if isinstance(doc, Document) else doc
        if re.match("^[A-Za-z0-9+/]{100,}={0,2}$", doc):
            b64_images.append(doc)
        else:
            texts_out.append(doc)

    return {"images" : b64_images, "texts" : texts_out}

# 3) 앞서 검색한 결과(found)로 동작 확인 → 이미지 / 텍스트 건수 출력

split_data = split_image_text_types(found)
print("이미지:" , len(split_data["images"]),"건")
print("표:" , len(split_data["texts"]),"건")

이미지: 1 건
표: 3 건


## 6. 최종 멀티모달 체인 만들기

체인의 구조는 세 칸입니다.

```
{검색 + 질문}  →  img_prompt_func  →  멀티모달 LLM  →  StrOutputParser
   재료 모으기      메시지로 조립        답변 생성        문자열로 꺼내기
```

**TODO 4~5.** 주석을 따라 체인을 작성하세요.

💡 **힌트**
- TODO 4: 6교시에서 Base64 앞에 붙였던 것과 같은 접두사입니다
- TODO 5: 위 그림의 3번 칸에 들어갈 변수 이름입니다

In [6]:
# ── 6. 최종 멀티모달 체인 만들기 ──────────────────────────
# 1) import
#    - HumanMessage (langchain_core.messages)
#    - StrOutputParser (langchain_core.output_parsers)
#    - ChatPromptTemplate (langchain_core.prompts)
#    - RunnableLambda, RunnablePassthrough (langchain_core.runnables)
#    - ChatOpenAI (langchain_openai)
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda , RunnablePassthrough
from langchain_openai import ChatOpenAI

# 2) img_prompt_func(data_dict) 함수 정의
#    """검색된 원본을 멀티모달 LLM이 읽을 형태로 조립합니다."""
#    - messages 빈 리스트 준비
#    - 1) 이미지를 먼저 넣습니다 (모델이 그림을 먼저 보게)
#         data_dict["context"]["images"] 마다 {"type": "image_url", "image_url": {"url": ...}} 추가
#         TODO 4: Base64 앞에 붙일 접두사를 완성하세요 (data:image/jpeg;base64,)
#    - 2) 텍스트 컨텍스트와 질문을 넣습니다
#         texts 를 줄바꿈으로 합치고 {"type": "text", "text": ...} 추가
#         지시문: "아래 자료를 근거로 질문에 한국어로 답하세요. 자료에 없는 내용은 모른다고 답하세요."
#         + 질문(data_dict["question"]) + 참고 텍스트
#    - [HumanMessage(content=messages)] 반환
def img_prompt_func(data_dict):
    messages = []

    # 1) 이미지를 먼저 넣는다
    for image in data_dict["context"]["images"]:
        messages.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{image}"},
        })

    # 2) 텍스트 컨텍스트와 질문을 넣는다 (for 문 밖)
    formatted_texts = "\n".join(data_dict["context"]["texts"])
    messages.append({
        "type": "text",
        "text": (
            "아래 자료를 근거로 질문에 한국어로 답하세요.\n"
            "자료에 없는 내용은 모른다고 답하세요.\n\n"
            f"질문: {data_dict['question']}\n\n"
            f"참고 텍스트:\n{formatted_texts}"
        ),
    })
    return [HumanMessage(content=messages)]

# 3) 모델: ChatOpenAI(temperature=0, model="gpt-4o-mini", max_tokens=1024)
model = ChatOpenAI(temperature=0, model="gpt-4o-mini", max_tokens=1024)
# 4) 체인 조립 (LCEL)
#    {"context": retriever | RunnableLambda(split_image_text_types),
#     "question": RunnablePassthrough()}
#    | RunnableLambda(img_prompt_func)
#    | TODO 5: 멀티모달 LLM 변수
#    | StrOutputParser()
chain = (
      {
            "context" : retriever | RunnableLambda(split_image_text_types),
            "question": RunnablePassthrough(),
      } | RunnableLambda(img_prompt_func)
         |model
         |StrOutputParser()
)

# 5) "최종 체인 조립 완료" 출력
print("mission complete!")

mission complete!


## 7. 질의 실행

In [7]:
# ── 7. 질의 실행 ──────────────────────────────────────────
# 1) 시작 시간 기록 (time.time())
start = time.time()

# 2) chain.invoke(query) 결과를 answer 에 저장

answer = chain.invoke(query)
# 3) 소요 시간, 질문(Q), 구분선, 답변 출력
print(f"답변생성 :{time.time() - start:.1f}초 \n ")
print("질문 :",query)
print("---"*60)
print(answer)

답변생성 :2.5초 
 
질문 : 과거 데이터 모의실험에서 상환 차수별 횟수와 빈도율은 어떻게 되나요? 1차 상환 비율은 몇 %인가요?
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
상환 차수별 횟수와 빈도율은 다음과 같습니다:

- 1차: 5009회 (97.21%)
- 2차: 80회 (1.55%)
- 3차: 64회 (1.24%)
- 4차: 0회 (0.00%)
- 5차: 0회 (0.00%)

1차 상환 비율은 97.21%입니다.


> ✅ **확인 포인트:** 답변에 **그래프나 표에서만 알 수 있는 내용**이 들어 있어야 합니다.
>
> 두루뭉술한 답이 나왔다면 질문을 더 구체적으로 바꿔보세요.

## 8. Before / After 비교 (가장 중요)

> 📌 **과제 채점 7점짜리 셀입니다. 반드시 하세요.**

같은 질문을 **텍스트 전용 RAG**와 **멀티모달 RAG**에 각각 던져 차이를 증거로 남깁니다.

먼저 비교용 텍스트 전용 RAG를 만듭니다. **이미지와 표를 뺀** 것 말고는 모든 조건이 같습니다.

In [8]:
# ── 8-1. 비교용 텍스트 전용 RAG ────────────────────────────
# 이미지와 표를 뺀 것 말고는 모든 조건이 같아야 합니다.

# 1) 텍스트만 들어간 벡터 DB 생성
#    - Chroma(collection_name="text_only_rag", embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"))
#    - text_2k_token 을 add_texts 로 저장
#    - as_retriever(search_kwargs={"k": 4}) 로 검색기 생성

vectorstore = Chroma(
    collection_name="text_only_rag", 
    embedding_function = OpenAIEmbeddings(model="text-embedding-3-small"),

)

vectorstore.add_texts(texts=text_2k_token)

text_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})



# 2) 프롬프트: ChatPromptTemplate.from_template
#    - 멀티모달 체인과 같은 지시문 + {question} + {context}
text_prompt = ChatPromptTemplate.from_template(
    "아래 자료를 근거로 질문에 한국어로 답하세요.\n"
    "자료에 없는 내용은 모른다고 답하세요.\n\n"
    "질문: {question}\n\n"
    "참고 텍스트:\n{context}"
)

# 3) 체인 조립
#    {"context": 검색기 | (문서들의 page_content 를 줄바꿈으로 합치는 함수),
#     "question": RunnablePassthrough()}
#    | 프롬프트 | model | StrOutputParser()
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

text_only_chain = (
    {
        "context": text_retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | text_prompt
    | model
    | StrOutputParser()
)

# 4) "비교용 텍스트 전용 RAG 준비 완료" 출력
print("비교용 텍스트 전용 RAG 준비 완료")

print(text_only_chain.invoke(query))

비교용 텍스트 전용 RAG 준비 완료


자료에는 과거 데이터 모의실험에서 상환 차수별 횟수와 빈도율에 대한 구체적인 내용이 포함되어 있지 않습니다. 또한 1차 상환 비율에 대한 정보도 제공되지 않았습니다. 따라서 이와 관련된 질문에 대한 답변은 드릴 수 없습니다.


In [9]:
# ── 8-2. Before / After 출력 ───────────────────────────────
# 1) 구분선과 질문 출력
print("-"*60)
print("질문: ",query)
# 2) [BEFORE] 텍스트 전용 RAG
#    - text_only_chain.invoke(query) 결과를 before 에 저장 후 출력
print("[BEFORE] 텍스트 전용 RAG")
before = text_only_chain.invoke(query)
print(before)

# 3) [AFTER] 멀티모달 RAG
#    - chain.invoke(query) 결과를 after 에 저장 후 출력
print("[AFTER] 멀티모달 RAG")
after = chain.invoke(query)
print(after)
# 4) "이 출력 전체를 캡처해 README.md에 넣으세요." 안내 출력

print("이 출력 전체를 캡쳐해 README.md에 넣으세요.")

------------------------------------------------------------
질문:  과거 데이터 모의실험에서 상환 차수별 횟수와 빈도율은 어떻게 되나요? 1차 상환 비율은 몇 %인가요?
[BEFORE] 텍스트 전용 RAG


자료에는 과거 데이터 모의실험에서 상환 차수별 횟수와 빈도율에 대한 구체적인 내용이 포함되어 있지 않습니다. 또한 1차 상환 비율에 대한 정보도 제공되지 않았습니다. 따라서 이와 관련된 질문에 대한 답변은 드릴 수 없습니다.
[AFTER] 멀티모달 RAG


상환 차수별 횟수와 빈도율은 다음과 같습니다:

- 1차: 5009회 (97.21%)
- 2차: 80회 (1.55%)
- 3차: 64회 (1.24%)
- 4차: 0회 (0.00%)
- 5차: 0회 (0.00%)

1차 상환 비율은 97.21%입니다.
이 출력 전체를 캡쳐해 README.md에 넣으세요.


**TODO 6. 두 답변의 차이를 적으세요 (더블클릭해서 편집)**

- 텍스트 전용 RAG의 답변은 어땠나요: 답변을 드릴 수 없다고 나왔습니다. 
- 멀티모달 RAG의 답변은 어땠나요: 구체적인 지표와 함께 상환 차수별 횟수와 빈도율이 정확하게 나왔습니다. 
- 차이가 생긴 이유는: 그 이유는 텍스트 전용 rag 는 이미지나 표를 인식하지 못하기 때문에 답변을 할 수가 없었고 멀티모달 rag 는 이미지와 표를 인식하기 때문에 정확한 답변을 할 수 있었습니다. 

> 💡 **차이가 잘 안 드러난다면** 질문이 잘못된 것입니다. 텍스트에도 같은 내용이 적혀 있을 가능성이 큽니다. **그래프에만 있는 정보**를 묻는 질문으로 바꿔 4번 셀부터 다시 돌리세요.

> 📸 **지금 이 출력을 스크린샷으로 저장하세요.** 포트폴리오에서 가장 강력한 자료입니다.

## 9. 번역 체인 통합 (선택)

번역만 로컬 모델에 맡겨 비용을 줄이는 구성입니다.

1. Ollama 설치

설치 파일

https://ollama.com/download 버전별 설치 파일(OllamaSetup.exe)을 받습니다.
masOS 터미널 실행하기:curl -fsSL https://ollama.com/install.sh | sh
windows powershell 에서 실행하기 :irm https://ollama.com/install.ps1 | iex
설치가 끝나면 작업 표시줄 트레이에 라마 아이콘이 뜹니다.

설치 후 새 터미널을 열어 확인합니다.
ollama --version

2. llama3.1:8b 모델 받기
powershell에서 실행 
ollama pull llama3.1:8b

용량이 약 4.9GB라서 네트워크에 따라 몇 분 걸립니다.
메모리는 8GB 이상을 권장합니다. 16GB면 여유가 있습니다.

3. 동작 테스트
powershell
ollama run llama3.1:8b "안녕하세요를 영어로 번역해줘"

답이 나오면 성공입니다. 대화 모드에서는 /bye로 빠져나옵니다.

4. 파이썬 패키지 설치

VS Code 터미널에서 가상환경(.venv)이 켜진 상태로 설치합니다.
powershell
pip install langchain-ollama

5. 용량확보하기 위해서는 실습 테스트 후 지워주세요


In [10]:
# ── 실습 후 용량 확보 (Ollama 정리) ───────────────────────
# 주석 해제 후 powershell 에서 실행
# ollama list              # 설치된 모델 확인
# ollama rm llama3.1:8b    # 삭제
# ollama list              # 목록에서 사라졌는지 확인

# 2. 파이썬 연동 패키지 삭제
# 가상환경(.venv)이 켜진 상태에서 실행합니다.
# %pip uninstall -y langchain-ollama ollama

# 삭제한 뒤에는 커널을 재시작하세요.

# 3. Ollama 프로그램까지 완전히 제거
# 설정 → 앱 → 설치된 앱 → Ollama → 제거를 선택합니다. winget uninstall Ollama.Ollama 로도 됩니다.
# 남은 모델 폴더를 지웁니다.

# powershell
# Remove-Item -Recurse -Force "$env:USERPROFILE\.ollama"

In [11]:
# ── 9-1. 노트북에서 Ollama 연결 확인 ──────────────────────
# 1) langchain_ollama.llms 의 OllamaLLM import
# 2) OllamaLLM(model="llama3.1:8b") 생성
# 3) "Translate to Korean: Hello, world" 를 invoke 해서 결과 출력

In [12]:
# ── 9-2. 번역 체인 통합 (선택) ────────────────────────────
# Ollama + llama3.1:8b 가 떠 있는 경우에만 실행하세요

# 1) 번역 프롬프트: ChatPromptTemplate.from_template
#    - "다음 영문 답변을 자연스러운 한국어로 번역하세요. 숫자와 고유명사는 그대로 두세요." + {answer}

# 2) translator = OllamaLLM(model="llama3.1:8b")

# 3) final_chain 조립
#    chain → (답변 문자열을 {"answer": 답변} 딕셔너리로 포장) → 번역 프롬프트 → translator → StrOutputParser()

# 4) final_chain.invoke(query) 결과 출력

---

## ✅ 체크포인트 4 — 모두 체크되면 완료

- [ v] 질문 하나에 대해 답변이 출력된다
- [ v] 답변에 **그래프나 표에서만 알 수 있는 내용**이 들어 있다
- [ v] 검색 결과에 이미지(Base64)가 최소 1건 포함된다
- [ v] Before/After 출력을 **캡처해 저장**했다
- [ ] 질의응답 5건 마크다운 표를 복사해 두었다
- [v ] TODO 1~6을 모두 채웠다

## 생각해 볼 질문 (답을 적어보세요)

**Q1. 요약과 원본에 같은 `doc_id`를 붙이지 않으면 어떤 일이 벌어질까요?**

답: 그렇게 되면 원본을 찾을 수 없게 됩니다. 결과가 비어 있거나 엉뚱한 원본이 연결될 수도 있습니다. 예를 들어서 표요약을 검색했는데 다른 이미지가 llm에게 전달되는 등 검색 자체로는 잘된것 처럼 보이지만 답변이 틀리거나 '정보가 없다'라는 답이 나와서 원인을 찾기 어렵게 됩니다.

**Q2. 검색은 요약으로 하는데, 왜 답변 생성에는 원본을 쓸까요? 요약만 쓰면 안 되나요?**

답: 요약만 쓰게 되면 근거가 부족해 정확한 답변이 나올수가 없습니다. 내용이 부실하게 나옵니다.

**Q3. `InMemoryStore`는 커널을 끄면 사라집니다. 실제 서비스로 만들려면 무엇을 바꿔야 할까요?**

답: 널을 꺼도 데이터가 남도록 docstore는 Redis·DB·파일 저장소로, vectorstore는 디스크에 저장되는 Chroma나 벡터 DB로 바꿉니다. 그리고 인덱싱은 미리 한 번만 하고 서비스는 저장된 인덱스를 불러와 쓰도록 분리합니다. doc_id는 다시 실행해도 바뀌지 않게 관리하고, 이미지는 파일 저장소에 두고 경로만 보관합니다.
## 자주 나는 에러

| 증상 | 원인 | 해결 |
| --- | --- | --- |
| `FileNotFoundError: summaries.pkl` |  저장 셀 미실행 | 앞 노트북 마지막 셀 재실행 |
| 검색 결과에 이미지가 없음 | 이미지 요약 품질이 낮음 | 프롬프트 보강 |
| 답변이 "자료에 없습니다" | 검색이 엉뚱한 원본을 가져옴 | 질문을 요약문 표현에 가깝게 수정 |
| 같은 원본이 여러 번 나옴 | `add_documents`를 두 번 실행 | 2번 셀부터 재실행 |
| `RateLimitError` | 질의 5건 연속 호출 | 잠시 대기 후 재실행 |
| Before/After 차이가 없음 | 질문이 텍스트로도 답 가능 | 그래프에만 있는 정보를 묻도록 수정 |
| `NameError: found` | 4번 셀을 안 돌림 | 위에서부터 순서대로 재실행 |

## 과제로 가져갈 것

| 산출물 | 어디서 나왔나 |
| --- | --- |
| Before/After 캡처 | 8번 셀 출력 |
| 질의응답 5건 표 | 10번 셀이 출력한 마크다운 |
| 아키텍처 다이어그램 | 직접 작도 (Excalidraw / Mermaid) |
| `README.md` | 문제 정의 + 위 자료들 |
| `.gitignore` | `.env`, `data/figures/`, `*.pdf` |

